## Using ONNX Runtime instead of PyTorch
Onnx can serve the same embedding models without the overhead of extra dependcies like NVIDIA and PyTorch which allow it be served a lighter weight production alternative.

The following packages were added:
```
uv add onnxruntime tokenizers numpy tqdm minsearch
uv add --dev huggingface-hub jupyter
```

Then we register a kernel for the project (basically tells jupyter to use this kernel):

`uv run python -m ipykernel install --user --name onnx-demo --display-name "onnx-demo"`

Then select the available kernel

In [1]:
# Embedding with no pytorch
from embedder import Embedder

embed = Embedder()

q1 = "Can I still join the course after the start date?"
q2 = "How to install Docker on Windows?"
d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."

v1 = embed.encode(q1)
v2 = embed.encode(q2)
dv = embed.encode(d)

In [ ]:
# computing cosine similiarities
v1.dot(dv)

np.float64(0.3233238425677314)

In [3]:
v2.dot(dv)

np.float64(0.01973045838992233)

In [4]:
# load docs
from ingest import load_faq_data

documents = load_faq_data()

In [5]:
# combine q + a to encode as one and check against encoded query
texts = [doc["question"] + " " + doc["answer"] for doc in documents]

In [6]:
# Embed in batches
from tqdm.auto import tqdm
import numpy as np

batch_size = 50
X = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = embed.encode_batch(batch)
    X.extend(batch_vectors)

X = np.array(X)

  0%|          | 0/27 [00:00<?, ?it/s]

In [7]:
# using search
query = "Can I still join the course after the start date?"
v_query = embed.encode(query)

scores = X.dot(v_query)
idx = np.argmax(scores)

documents[idx]

{'id': '3f1424af17',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: Can I still join the course after the start date?',
 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute."}

In [8]:
scores

array([ 0.50192222,  0.26513016,  0.76294113, ..., -0.08384743,
        0.037598  , -0.12702173], shape=(1350,))